In [1]:
from componergy.preprocessing.generation_eia import main as build_generation
from componergy.preprocessing.demand_eia861m import main as build_demand
build_generation(); build_demand()

/mnt/Workspace/Work/componergy/data/processed/california_generation_monthly.csv already exists, skipping (pass force=True to rebuild).
reading /mnt/Workspace/Work/componergy/data/raw/electricity/HS861M_1990-2009.xlsx (sheet 'Monthly')...
reading /mnt/Workspace/Work/componergy/data/raw/electricity/HS861M_2010-.xlsx (sheet 'Monthly-States')...
combined: 428 months, 1990-01-01 to 2025-08-01
wrote /mnt/Workspace/Work/componergy/data/processed/california_demand_monthly.csv


In [2]:
from componergy.paths import COUNTY_CONSUMPTION_XLSX, DEMAND_MONTHLY_FILE, GENERATION_MONTHLY_FILE
from componergy.preprocessing.consumption_cec import load_raw_consumption, statewide_monthly_mwh
import pandas as pd
cec = statewide_monthly_mwh(load_raw_consumption(COUNTY_CONSUMPTION_XLSX)) if COUNTY_CONSUMPTION_XLSX.exists() else None
eia = pd.read_csv(DEMAND_MONTHLY_FILE, parse_dates=[0], index_col=0)['total_sales']
gen = pd.read_csv(GENERATION_MONTHLY_FILE, parse_dates=[0], index_col=0)['Total']
print('CEC:', None if cec is None else (cec.index.min().date(), cec.index.max().date()), '| EIA-861M:', eia.index.min().date(), eia.index.max().date())

CEC: (datetime.date(2008, 1, 1), datetime.date(2024, 12, 1)) | EIA-861M: 1990-01-01 2025-08-01


In [3]:
from componergy.analysis.demand_crosscheck import compare_demand_sources
from componergy.analysis.system_boundary import validate_system_boundary
if cec is not None:
    print(compare_demand_sources(cec, eia)['verdict'])
for name, d in (('CEC', cec), ('EIA-861M', eia)):
    if d is not None:
        r = validate_system_boundary(d / 1000, gen / 1000); print(name, round(r['median_import_fraction'], 3), r['verdict'][:90])

weak agreement (anomaly correlation 0.59): the two series measure materially different things, so the source choice must be justified explicitly and results shown for both
CEC 0.289 median implied net-import fraction falls within the expected range for California; treatin
EIA-861M 0.218 median implied net-import fraction falls within the expected range for California; treatin
